# From FITS to Zarr, with an agent

**Block 3, Agent-Driven Research Software Engineering**

Radio telescopes hand you images as FITS files, often one file per frequency
channel. That is fine for looking at one image and painful for anything else:
slicing a spectrum through 26 files, loading only the part of the sky you care
about, or sharing a cube with a collaborator. **Zarr** (read through `xarray`)
stores the whole cube as one chunked, lazily loaded dataset, so you can slice it
without reading everything into memory.

This is not a toy problem. The cube here is a **forward-modeled DSA**
observation: a simulated sky passed through the instrument model, the way survey
teams exercise their pipelines before real data arrive. We convert it with a
production FITS-to-Zarr converter that radio teams already use. Converting is
the easy part. The hard part is **knowing the conversion is right**: a
transposed axis, a dropped channel, or a sky position shifted by a pixel all
look fine until someone does science on them. Once it is right, the payoff is
that science gets easy: at the end we integrate the whole cube in one line.

In this notebook you drive an agent through the same loop as the rest of Block 3:

| Step | What you do | Skill | Artifact in `demo/docs/` |
|---|---|---|---|
| 1. Profile | Find out what the FITS files contain | `profile-dataset` | `profile-dsa-cube.md` |
| 2. Convert | Turn 26 FITS files into one Zarr cube | `fits-to-zarr` | `convert-dsa-cube.md` |
| 3. Verify | Check the Zarr against the FITS, and check the agent | `fits-to-zarr` (`verify.py`) | `verify-dsa-cube.md` |
| 4. Use | Integrate the cube, then put it on the sky | xarray, astrowidget | (in this notebook) |

**Where the agent lives.** On the workshop JupyterHub, open a terminal and run
`claude` from the repo root (Claude Code). In a Codespace, use Copilot Chat in
**Agent** mode. Either way, the agent finds the same skills: they are plain
markdown files in `.github/skills/`.

**The data** is a forward-modeled spectral-line cube for DSA (the Deep Synoptic
Array): 26 channels between 1415.2 and 1418.5 MHz, just below the 1420.4 MHz
HI line. It is simulated, so treat any number you derive as a check of the
method, not a result.

## 0. Setup

We find the repo root, the input FITS folder, and where outputs go. The FITS
folder is, in order:

1. the `FITS_DATA_DIR` environment variable, if set;
2. on the workshop JupyterHub, the shared S3-backed folder
   `~/s3/schmidt-observatory-system/nov26-symposium/` (mounted read-only into
   every server, so nobody downloads anything);
3. `.data/TSOS_Science_Symposium/` at the repo root, for a local copy.

Nothing in this notebook ever writes to that folder. Outputs go under the demo
folder in your own copy of the repo.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from astropy.io import fits
from astropy.wcs import WCS
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "blocks").exists():
            return path
    raise RuntimeError("Run this notebook from inside the workshop repo.")


def find_data_dir(repo: Path) -> Path:
    if os.environ.get("FITS_DATA_DIR"):
        return Path(os.environ["FITS_DATA_DIR"]).expanduser()
    hub_share = Path.home() / "s3" / "schmidt-observatory-system" / "nov26-symposium"
    for root in (hub_share, repo / ".data"):
        hits = sorted(root.glob("**/mockfm_nl_cube_ch00_*.fits")) if root.is_dir() else []
        if hits:
            return hits[0].parent
    return repo / ".data" / "TSOS_Science_Symposium"


REPO = find_repo_root(Path.cwd())
DATA_DIR = find_data_dir(REPO)
DEMO = REPO / "blocks" / "03-research-loop" / "demo"
DOCS = DEMO / "docs"
OUT = DEMO / "output"
ZARR = OUT / "dsa-cube.zarr"
SKILL = REPO / ".github" / "skills" / "fits-to-zarr" / "scripts"

def rel(path: Path) -> str:
    # Repo-relative when possible, so committed artifacts carry no local paths.
    try:
        return str(path.resolve().relative_to(REPO))
    except ValueError:
        return str(path)


FITS_FILES = sorted(DATA_DIR.glob("*.fits"))
if not FITS_FILES:
    raise FileNotFoundError(
        f"No FITS files in {DATA_DIR}. Set FITS_DATA_DIR or see demo/starter/README.md."
    )
print(f"{len(FITS_FILES)} FITS files in {rel(DATA_DIR)}")
print(f"Zarr will be written to {ZARR.relative_to(REPO)}")

## 1. Meet the data

### Demo 1: `profile-dataset`

Before converting anything, have the agent describe what it is working with.
Paste this into the agent:

```
Read blocks/03-research-loop/demo/AGENTS.md first, then use the profile-dataset skill on the FITS folder it describes.
```

**Watch for:** `demo/docs/profile-dsa-cube.md`. A good profile reads the FITS
*headers* (it does not need the pixels) and notices what varies across files.

While it works, look at the headers yourself. One row per file, a few keywords
that decide how a conversion has to work:

In [ ]:
rows = []
for path in FITS_FILES:
    h = fits.getheader(path)
    rows.append({
        "file": path.name,
        "size": f"{h['NAXIS1']} x {h['NAXIS2']}",
        "freq_MHz": h["CRVAL3"] / 1e6,
        "DATE-OBS": h["DATE-OBS"],
        "RADESYS": h.get("RADESYS"),
        "EQUINOX": h.get("EQUINOX", "(missing)"),
        "BMAJ_arcsec": h["BMAJ"] * 3600,
    })
headers = pd.DataFrame(rows)
headers.head()

Three things in that table matter for the conversion, and they are exactly what
an agent tends to get wrong without guidance:

- **The frequency lives in the header, not usefully in the filename.** The
  filenames carry the frequency with a decimal point, which the OVRO-LWA
  converter's filename parser does not expect. Group by header values.
- **Every file has the same `DATE-OBS`.** This is one snapshot across 26
  channels, so the cube has one time step and 26 frequency planes.
- **There is no `EQUINOX`.** The frame is ICRS, which does not need one, but the
  converter relabels the frame as FK5, and FK5 does. The skill's script handles
  this; an agent improvising its own conversion will crash here or, worse, quietly
  guess.

Here is one channel on the sky. The axes come from the file's WCS (World
Coordinate System), which maps each pixel to a right ascension and declination:

In [ ]:
channel = 13
with fits.open(FITS_FILES[channel]) as hdul:
    image = np.squeeze(hdul[0].data)
    wcs = WCS(hdul[0].header).celestial

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection=wcs)
ax.imshow(image, origin="lower", cmap="inferno", vmin=0, vmax=np.nanpercentile(image, 99.5))
ax.set_xlabel("Right ascension")
ax.set_ylabel("Declination")
ax.set_title(f"{FITS_FILES[channel].name}")
plt.show()

## 2. Convert with the skill

### Demo 2: `fits-to-zarr`

```
Use the fits-to-zarr skill to convert the FITS folder from AGENTS.md into blocks/03-research-loop/demo/output/dsa-cube.zarr, verify it, and write the conversion record.
```

**Watch for:**

- Does the agent **run the skill's scripts** (`convert.py`, then `verify.py`) or
  start writing its own converter? The skill says not to. An agent ignoring its
  instructions is a failure mode you should catch and push back on.
- Does it **report the verification honestly**, including the small frame
  relabel the converter introduces?
- Artifacts: `demo/docs/convert-dsa-cube.md` and `demo/docs/verify-dsa-cube.md`.

The skill is just a folder: `SKILL.md` (instructions the agent reads) plus two
scripts that do the deterministic work. Take a look:

In [ ]:
print((SKILL.parent / "SKILL.md").read_text()[:1500])

**No agent, or did it go sideways?** Run the skill's conversion script yourself.
This is the same command the skill tells the agent to run. The cell skips the
work if a Zarr store already exists (from the agent, or an earlier run); set
`FORCE = True` to rebuild it.

In [ ]:
FORCE = False

if FORCE or not ZARR.exists():
    result = subprocess.run(
        [sys.executable, rel(SKILL / "convert.py"), rel(DATA_DIR), rel(OUT), "--zarr-name", ZARR.name],
        capture_output=True, text=True, cwd=REPO,
    )
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError("convert.py failed; read the error above.")
    summary = json.loads(result.stdout.strip().splitlines()[-1])
    print(json.dumps(summary, indent=2))
else:
    print(f"Using existing {ZARR.relative_to(REPO)}")

Open the result. Notice the five dimensions: one `time`, 26 `frequency` planes,
one `polarization`, and the two sky axes `l` and `m`. The data is not loaded yet:
xarray read only the metadata, and each chunk is loaded when you use it.

In [ ]:
cube = xr.open_zarr(ZARR)
cube

## 3. Trust, but verify

An agent's "Done! The conversion succeeded." is a **claim**. The evidence is a
check you can re-run. The skill's `verify.py` compares the Zarr against the
**original** FITS files: every pixel of every channel, the frequencies and time,
the beam, and where each pixel lands on the sky.

### Demo 3: read the evidence

```
Show me the verification evidence for the conversion. Which checks ran, what tolerance did the sky-position check use, and what was the worst offset?
```

You can also run the verifier yourself:

In [ ]:
DOCS.mkdir(parents=True, exist_ok=True)
check = subprocess.run(
    [sys.executable, rel(SKILL / "verify.py"), rel(DATA_DIR), rel(ZARR),
     "--report", rel(DOCS / "verify-dsa-cube.md")],
    capture_output=True, text=True, cwd=REPO,
)
display(Markdown((DOCS / "verify-dsa-cube.md").read_text()))
print("exit code:", check.returncode)

Do not stop at the script either: **spot-check one thing yourself**, by a route
the script did not take. Pick a random channel and pixel and compare the Zarr
value with the FITS value.

One detail trips up almost everyone (agents included): a FITS image is stored as
`(y, x)`, while the Zarr cube is `(l, m)` with `l` along x. So the Zarr plane is
the FITS image **transposed**, and Zarr pixel `[x, y]` is FITS pixel `[y, x]`.
We fix the random seed so everyone checks the same pixel.

In [ ]:
rng = np.random.default_rng(seed=42)
freqs = np.array([fits.getheader(f)["CRVAL3"] for f in FITS_FILES])
i_file = int(rng.integers(len(FITS_FILES)))
x, y = int(rng.integers(cube.sizes["l"])), int(rng.integers(cube.sizes["m"]))

fits_value = np.squeeze(fits.getdata(FITS_FILES[i_file]))[y, x]
zarr_value = cube["SKY"].sel(frequency=freqs[i_file], method="nearest").isel(time=0, polarization=0, l=x, m=y).values.item()
print(f"{FITS_FILES[i_file].name}, pixel x={x}, y={y}")
print(f"FITS: {fits_value!r}   Zarr: {zarr_value!r}   equal: {fits_value == zarr_value}")

Now the sky position of that same pixel, from the FITS WCS and from the
`right_ascension` / `declination` coordinates stored in the Zarr. The converter
labels the frame FK5 (J2000) while the files say ICRS, so expect a tiny offset:
a few hundredths of an arcsecond, far below the 1 arcsecond pixel size. That is
a real, documented deviation. A good verification measures it and says so,
rather than hiding it or panicking about it.

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord

from_fits = WCS(fits.getheader(FITS_FILES[i_file])).celestial.pixel_to_world(x, y)
ra = cube["right_ascension"].isel(l=x, m=y).values.item()
dec = cube["declination"].isel(l=x, m=y).values.item()
from_zarr = SkyCoord(ra * u.deg, dec * u.deg, frame=cube["right_ascension"].attrs.get("frame", "fk5"))
print("FITS:", from_fits.icrs.to_string("hmsdms"))
print("Zarr:", from_zarr.icrs.to_string("hmsdms"))
print(f"offset: {from_fits.separation(from_zarr).to(u.arcsec):.4f}")

## 4. Use the cube

Everything so far was about earning trust in one object, `cube`. Now use it.
Because it is one labelled array, a reduction along any axis is one line, and
dask computes it chunk by chunk instead of loading 26 files. Two reductions tell
most of the story of a spectral-line cube: sum over **frequency** for a map, sum
over **sky** for a spectrum.

The same lines work unchanged on a Zarr store in object storage
(`xr.open_zarr("s3://...")`). The computation then runs on the hub, in the same
cloud as the data, and reads only the chunks a reduction touches; nothing comes
to your laptop.

First, a velocity axis. The cube's spectral frame is LSRK (`SPECSYS`); we use the
radio convention, which keeps channels uniform in velocity, and the HI rest
frequency:

In [ ]:
from astropy import units as u

HI = 1420.405751768 * u.MHz
velocity = (cube["frequency"].values * u.Hz).to(u.km / u.s, equivalencies=u.doppler_radio(HI))
dv = np.abs(np.diff(velocity)).mean()
cube = cube.assign_coords(velocity=("frequency", velocity.value))
print(f"v_radio (LSRK): {velocity.min():.0f} to {velocity.max():.0f}, channel width {dv:.2f}")

### Integrated intensity (moment 0)

$M_0 = \sum_\nu S_\nu \, \Delta v$, in Jy/beam km/s. Build it and look at what
comes back *before* computing: a dask graph, not pixels.

What to look for: the **whole ring** of emission. That is atomic hydrogen (HI)
gas in the disk of the galaxy, summed over every velocity.

No continuum subtraction here: the line fills almost the whole band, leaving
few line-free channels to fit a continuum from. So the brightest spot in the
map is not the galaxy: it is an unrelated continuum point source near it,
integrated over 26 channels, and it produces some artifacts around it. Fine
for a look; for a measurement you would subtract it first.

In [ ]:
mom0_lazy = (cube["SKY"] * dv.value).sum("frequency")
print(mom0_lazy.data)  # a recipe: no pixels read yet

mom0 = mom0_lazy.compute()
mom0.attrs["units"] = "Jy/beam km/s"
print(f"computed {mom0.shape}, peak {float(mom0.max()):.3f} {mom0.attrs['units']}")

### Integrated spectrum

Now sum over the sky instead. Pixels are in Jy/beam, so the pixel sum divided by
the beam area in pixels is a flux density in Jy. The beam comes from the cube's
`BEAM` variable and the pixel scale from its WCS: everything from the Zarr, no
FITS needed. The continuum level is estimated from the band-edge channels and
removed.

In [ ]:
from astrowidget.wcs import get_wcs

zarr_wcs = get_wcs(cube)
pix_arcsec = abs(zarr_wcs.wcs.cdelt[0]) * 3600
beam = cube["BEAM"].isel(time=0, polarization=0)
beam_unit = u.Unit(beam.attrs.get("units", "rad"))
bmaj = (beam.sel(beam_param="major").values * beam_unit).to_value(u.arcsec)
bmin = (beam.sel(beam_param="minor").values * beam_unit).to_value(u.arcsec)
beam_pix = np.pi * bmaj * bmin / (4 * np.log(2)) / pix_arcsec**2

spectrum = cube["SKY"].isel(time=0, polarization=0).sum(["l", "m"]).compute().values / beam_pix
line_free = [0, 1, 24, 25]
spectrum = spectrum - spectrum[line_free].mean()
line_flux = spectrum.sum() * dv.value

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.step(cube["velocity"], spectrum, where="mid")
ax.plot(cube["velocity"][line_free], spectrum[line_free], "o", label="continuum level from these")
ax.axhline(0, color="0.6", lw=0.8)
ax.set_xlabel("Radio velocity, LSRK (km/s)")
ax.set_ylabel("Flux density (Jy)")
ax.set_title(f"Whole-field spectrum, integrated line flux {line_flux:.0f} Jy km/s")
ax.legend()
plt.show()

Two horns with a dip between them: the classic profile of a rotating disk,
where gas on the near and far sides of the rotation piles up at the extreme
velocities. Keep that in mind for the channel maps below.

### Demo 4: astrowidget

[astrowidget](https://github.com/ovro-lwa/astrowidget) projects a radio image
onto the celestial sphere straight from Zarr, with no detour back through FITS,
using the WCS that `convert.py` attached to the cube. Behind it we put the
Digitized Sky Survey (DSS), loaded as HiPS tiles by Aladin Lite, so you see where
the field sits on the real sky. (The survey tiles load from the internet.)

astrowidget displays cubes shaped `(time, frequency, polarization, l, m)`, so we
wrap the moment-0 map as a one-channel cube and give it the same WCS.
Drag to pan, scroll to zoom out. Look for the HI ring of the disk, and the
unrelated point source beside it.

In [ ]:
from astrowidget import SkyWidget

mom0_cube = mom0.expand_dims(frequency=[float(cube["frequency"].mean())], axis=1).to_dataset(name="MOM0")
mom0_cube["MOM0"].attrs["fits_wcs_header"] = cube["SKY"].attrs["fits_wcs_header"]

sky = SkyWidget()
sky.background_survey = "DSS"  # set before display: the survey loads when the widget renders
sky.opacity = 0.8
sky.set_dataset(mom0_cube, var="MOM0")
sky.colormap = "inferno"
sky.stretch = "sqrt"
sky.auto_scale(50, 99.9)
sky

Now look inside the cube. Point the same widget back at the full cube and step
from one horn of the spectrum to the other. Watch the emission walk along the
disk as the velocity changes: that is the rotation. (Changing the slice resets
the colour scale, so we set it again each time.)

In [ ]:
sky.set_dataset(cube)
sky.update_slice(0, 4)
sky.auto_scale(50, 99.9)
print(f"channel 4: {float(cube['velocity'][4]):.0f} km/s")

In [ ]:
sky.update_slice(0, 21)
sky.auto_scale(50, 99.9)
print(f"channel 21: {float(cube['velocity'][21]):.0f} km/s")

**If the widget does not render** (it needs a browser with WebGL2), here are the
same three views as static images, drawn from the Zarr with its stored WCS:

In [ ]:
views = [
    (mom0.isel(time=0, polarization=0).values.T, f"Moment 0 ({mom0.attrs['units']})"),
    *[
        (cube["SKY"].isel(time=0, frequency=k, polarization=0).values.T,
         f"Channel {k}: {float(cube['velocity'][k]):.0f} km/s")
        for k in (4, 21)
    ],
]
fig = plt.figure(figsize=(15, 5))
for i, (image, title) in enumerate(views, start=1):
    ax = fig.add_subplot(1, 3, i, projection=zarr_wcs)
    ax.imshow(image, origin="lower", cmap="inferno",
              vmin=np.nanpercentile(image, 50), vmax=np.nanpercentile(image, 99.9))
    ax.set_xlabel("Right ascension")
    ax.set_ylabel("Declination" if i == 1 else " ")
    ax.set_title(title)
plt.show()

**Stretch goals: describe it, don't code it.** Ask the agent, then read what it
wrote before you run it:

```
In blocks/03-research-loop/demo/notebook.ipynb, add a cell that shows channel 5 of the Zarr cube in astrowidget with the viridis colormap, zoomed to a 0.1 degree field of view around the phase centre.
```

```
Using line_flux and the velocity axis in blocks/03-research-loop/demo/notebook.ipynb, estimate the HI mass assuming the source is in the Hubble flow at its systemic velocity. State every assumption and constant you use.
```

For the second, check its work: the standard relation is
$M_{\rm HI} = 2.356\times10^{5}\,D_{\rm Mpc}^2 \int S\,dv$ (Jy km/s). Did it
pick a sensible systemic velocity and $H_0$, and say so?

## Wrap-up

Commit the audit trail, as with any agent-driven workflow:

```bash
git add blocks/03-research-loop/demo/docs/ && git commit -m "docs: FITS to Zarr conversion record"
```

The Zarr itself stays out of git (it is regenerated in seconds); the record of
**what was converted and how it was checked** goes in.

**Bridge to Block 4.** `fits-to-zarr` is a markdown file plus two scripts. In
Block 4 you will write a skill like it for a task from your own work.